# Globale Konfiguration

Die Klasse `ElectionConfig` enthält alle Parameter, die für eine gesamte Wahl gelten. Dieses Notebook zeigt, welche Parameter es gibt, was sie bewirken und wie man sie setzt.

Für eine detaillierte Erklärung der Konzepte siehe [Globale Konfiguration](../../docs/source/de/globale_konfiguration.md).

In [ ]:
import pandas as pd
from ipres import ElectionConfig, ConstituenciesConfig, SuperMajorityMargin, MarginUnit
from ipres.election_config import ConstituencyRepresentation, DrawLotsStrategy

## Pflichtparameter

### `constituencies_config` — Wahlkreise

Die Wahlkreise werden als DataFrame übergeben. Jede Zeile ist ein Wahlkreis mit Namen und Größe (Anzahl der Wahlberechtigten).

In [ ]:
wahlkreise_df = pd.DataFrame({
    'constituency_name': ['Nord', 'Süd', 'Ost', 'West', 'Mitte'],
    'constituency_size': [80_000, 60_000, 70_000, 90_000, 50_000],
})

cc = ConstituenciesConfig.from_dataframe(wahlkreise_df)
print(f"Anzahl Wahlkreise: {cc.getNumberOfConstituencies()}")

### `participating_parties` — Parteien

Eine einfache Liste von Parteinamen.

In [ ]:
parteien = ['A', 'B', 'C', 'D']

config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
)
print(f"Parteien: {config.participating_parties}")

---

## `parliament_majority_margin` — Regierungsabstand

Der Regierungsabstand gibt an, wie viel mehr als 50 % die Regierung haben soll — als Puffer, damit die Mehrheit auch bei Krankheit oder Enthaltungen erhalten bleibt.

Er kann entweder in **Prozent** oder in **Sitzen** angegeben werden. `ElectionConfig` rechnet immer in beide Einheiten um.

In [ ]:
# Angabe in Prozent: Regierung braucht 55 % der Sitze
config_5pct = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
)

# Angabe in Sitzen: Regierung braucht 3 Sitze mehr als die Hälfte
config_3seats = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    parliament_majority_margin=SuperMajorityMargin(3, MarginUnit.SEATS),
)

print(f"Parlamentssitze: {config_5pct.parliamentarySeats}")
print()
print("Angabe in Prozent (5 %):")
print(f"  Mehrheitsschwelle: {config_5pct.getParliamentMajorityPercent():.1f} %")
print(f"  Mehrheitsschwelle: {config_5pct.getParliamentMajoritySeats()} Sitze")
print()
print("Angabe in Sitzen (3 Sitze):")
print(f"  Mehrheitsschwelle: {config_3seats.getParliamentMajorityPercent():.1f} %")
print(f"  Mehrheitsschwelle: {config_3seats.getParliamentMajoritySeats()} Sitze")

## `constituency_representation` — Wahlkreisrepräsentation

Dieser Parameter bestimmt, welche Parteien Wahlkreise zugeteilt bekommen, und hat direkte Auswirkung auf die **Größe des Parlaments**.

- **`ENTIRE_PARLIAMENT`** *(Standard)*: Alle Parteien erhalten Wahlkreise proportional zu ihrer Sitzzahl. Die Parlamentsgröße berechnet sich einfach als `2 × Anzahl Wahlkreise`.

- **`GOVERNING_MAJORITY`**: Nur die Regierungsparteien erhalten Wahlkreise. Damit jeder Wahlkreis in der Regierungsfraktion vertreten ist, muss das Parlament größer sein: Die Gesamtzahl der Parlamentssitze wird so gewählt, dass die Regierungsmehrheit genau `2 × Anzahl Wahlkreise` Sitze umfasst.

In [ ]:
margin = SuperMajorityMargin(5.0, MarginUnit.PERCENT)  # 55 % Schwelle

config_ep = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    parliament_majority_margin=margin,
    constituency_representation=ConstituencyRepresentation.ENTIRE_PARLIAMENT,
)

config_gm = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    parliament_majority_margin=margin,
    constituency_representation=ConstituencyRepresentation.GOVERNING_MAJORITY,
)

n = cc.getNumberOfConstituencies()
print(f"Wahlkreise:                        {n}")
print()
print(f"{'':35} ENTIRE_PARLIAMENT   GOVERNING_MAJORITY")
print(f"{'Parlamentssitze':35} {config_ep.parliamentarySeats:<20} {config_gm.parliamentarySeats}")
print(f"{'Regierungsmehrheit (Sitze)':35} {config_ep.getParliamentMajoritySeats():<20} {config_gm.getParliamentMajoritySeats()}")
print(f"{'Regierungsmehrheit (%)':35} {config_ep.getParliamentMajorityPercent():<20.1f} {config_gm.getParliamentMajorityPercent():.1f}")
print(f"{'Regierungssitze = 2 × Wahlkreise':35} {config_ep.getParliamentMajoritySeats() == 2*n!s:<20} {config_gm.getParliamentMajoritySeats() == 2*n}")

Bei `GOVERNING_MAJORITY` ist das Parlament größer, weil die Regierungsmehrheit mindestens `2 × Wahlkreise` Sitze umfassen muss. Bei `ENTIRE_PARLIAMENT` gilt diese Eigenschaft nicht — dort ist die Parlamentsgröße einfach fix.

### Auswirkung des Regierungsabstands auf die Parlamentsgröße (nur bei `GOVERNING_MAJORITY`)

Je größer der Regierungsabstand, desto größer muss das Parlament sein:

In [ ]:
rows = []
for abstand_pct in [0, 2, 5, 10, 15, 20]:
    cfg = ElectionConfig(
        constituencies_config=cc,
        participating_parties=parteien,
        parliament_majority_margin=SuperMajorityMargin(abstand_pct, MarginUnit.PERCENT),
        constituency_representation=ConstituencyRepresentation.GOVERNING_MAJORITY,
    )
    rows.append({
        'Regierungsabstand (%)': abstand_pct,
        'Mehrheitsschwelle (%)': cfg.getParliamentMajorityPercent(),
        'Parlamentssitze': cfg.parliamentarySeats,
        'Regierungssitze': cfg.getParliamentMajoritySeats(),
    })

pd.DataFrame(rows).set_index('Regierungsabstand (%)')

---

## `draw_lots_strategy` — Gleichstandsauflösung

Wenn bei der Sitzzuteilung ein Gleichstand nicht eindeutig aufgelöst werden kann, entscheidet diese Strategie:

- **`RANDOM`** *(Standard)*: Zufällige Entscheidung (beeinflusst durch `seed`).
- **`MARGINAL_LEAD`**: Der marginale Stimmunterschied gilt als zufällig entstanden — die Partei, die zufällig etwas mehr Stimmen erhalten hat, gewinnt.

In [ ]:
config_random = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    draw_lots_strategy=DrawLotsStrategy.RANDOM,
)

config_marginal_lead = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
)

print(f"RANDOM:        {config_random.draw_lots_strategy}")
print(f"MARGINAL_LEAD: {config_marginal_lead.draw_lots_strategy}")

---

## `seed` — Reproduzierbarkeit

Mit `seed` lässt sich der Zufallsgenerator auf einen festen Startwert setzen, sodass Simulationen reproduzierbar sind. Der Standardwert `None` bedeutet, dass der Startwert zufällig gewählt wird.

In [ ]:
# Ohne seed: jedes Mal ein anderes Ergebnis bei Zufallsentscheidungen
config_no_seed = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
)

# Mit seed: reproduzierbar
config_seeded = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    seed=42,
)

print(f"Kein seed: {config_no_seed.seed}")
print(f"Mit seed:  {config_seeded.seed}")

---

## `ballot_majority_margin` — Wahlgangschwelle

Der `ballot_majority_margin` ist der Mindestabstand über 50 %, den ein Kandidat in einem **einzelnen Wahlgang** erreichen muss, um diesen Wahlgang zu gewinnen. Er ist unabhängig von der `parliament_majority_margin`, die die Sitzverteilung im Parlament regelt.

Wie `parliament_majority_margin` kann er in **Prozent** oder in **Sitzen** angegeben werden. Standard: 2 % (d. h. 52 %).

In [ ]:
config_ballot = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    ballot_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
)

print(f"Wahlgangschwelle:   {config_ballot.getBallotMajorityPercent():.1f} %  (Kandidat muss diesen Anteil erreichen, um den Wahlgang zu gewinnen)")
print(f"Parlamentsschwelle: {config_ballot.getParliamentMajorityPercent():.1f} %  (Regierung muss diesen Sitzanteil im Parlament halten)")

---

## `language` — Ausgabesprache

Mit `language` wird die Sprache für alle Ausgaben festgelegt: Tabellenbeschriftungen, Spaltenköpfe, Diagrammtitel und Zahlenformatierung.

- **`Language.DE`** *(Standard)*: Deutsche Ausgabe.
- **`Language.EN`**: Englische Ausgabe.

In [ ]:
from ipres.election_config import Language

config_de = ElectionConfig(constituencies_config=cc, participating_parties=parteien, language=Language.DE)
config_en = ElectionConfig(constituencies_config=cc, participating_parties=parteien, language=Language.EN)

print(f"Deutsch:  {config_de.language}")
print(f"Englisch: {config_en.language}")

---

## `seat_distribution_method` — Sitzverteilungsmethode

Diese Einstellung bestimmt die Methode der proportionalen Sitzverteilung, die bei der Auswertung einer Wahl verwendet wird.

- **`SAINTE_LAGUE`** *(Standard)*: Sainte-Laguë-Verfahren (auch Schepers-Verfahren). Bevorzugt keine Parteigrößen systematisch.
- **`D_HONDT`**: D'Hondt-Verfahren. Bevorzugt tendenziell größere Parteien.
- **`HARE_NIEMEYER`**: Hare-Niemeyer-Verfahren (Methode der größten Reste). Bevorzugt tendenziell kleinere Parteien.

Die Evaluatorklassen (`SeatDistributor`, `ElectionEvaluator`) können diesen Wert für eine einzelne Auswertung überschreiben.

In [ ]:
from ipres.election_config import SeatDistributionMethod

for method in SeatDistributionMethod:
    cfg = ElectionConfig(
        constituencies_config=cc,
        participating_parties=parteien,
        seat_distribution_method=method,
    )
    print(f"seat_distribution_method = {cfg.seat_distribution_method}")

---

## `quota_correction_strategy` — Quotenkorrekturstrategie

Bei der Sitzverteilung ergeben sich für die Wahlkreiszuteilung ganzzahlige Quoten (Sitze ÷ 2). Da die Summe dieser Quoten nicht immer exakt der Wahlkreisanzahl entspricht, wird eine Korrektur vorgenommen. Diese Einstellung bestimmt die Strategie dafür.

- **`FAVOR_LARGE_PARTIES`** *(Standard)*: Zusatzwahlkreise gehen an die größten Parteien.
- **`FAVOR_SMALL_PARTIES`**: Zusatzwahlkreise gehen an die kleinsten Parteien.
- **`PROPORTIONAL`**: Zusatzwahlkreise werden proportional zur Sitzzahl verteilt.
- **`PROPORTIONAL_REVERSED`**: Wie `PROPORTIONAL`, aber in umgekehrter Reihenfolge.
- **`RANDOM`**: Zufällige Vergabe der Zusatzwahlkreise.
- **`NEGOTIATED`**: Verteilung wird durch einen Callback festgelegt.

Die Evaluatorklassen (`ConstituencyCountDeterminer`, `ElectionEvaluator`) können diesen Wert für eine einzelne Auswertung überschreiben.

In [ ]:
from ipres.election_config import QuotaCorrectionStrategy

for strategy in [QuotaCorrectionStrategy.FAVOR_LARGE_PARTIES, QuotaCorrectionStrategy.FAVOR_SMALL_PARTIES]:
    cfg = ElectionConfig(
        constituencies_config=cc,
        participating_parties=parteien,
        quota_correction_strategy=strategy,
    )
    print(f"quota_correction_strategy = {cfg.quota_correction_strategy}")

---

## `constituency_allocation_method` — Wahlkreiszuweisungsalgorithmus

Dieser Parameter bestimmt den Algorithmus, mit dem Wahlkreise den Parteien zugewiesen werden, nachdem die Anzahl der Wahlkreise pro Partei feststeht.

- **`OPTIMAL`** *(Standard)*: Optimale Zuweisung, die den Gesamtnutzen maximiert.
- **`GREEDY`**: Gieriger Algorithmus — schnell, aber nicht garantiert optimal.
- **`STABLE_MATCHING`**: Stabile Paarung nach Gale-Shapley.

Die Evaluatorklassen (`ConstituencyAssigner`, `ElectionEvaluator`) können diesen Wert für eine einzelne Auswertung überschreiben.

In [ ]:
from ipres.allocation import ConstituencyAllocationMethod

for method in ConstituencyAllocationMethod:
    cfg = ElectionConfig(
        constituencies_config=cc,
        participating_parties=parteien,
        constituency_allocation_method=method,
    )
    print(f"constituency_allocation_method = {cfg.constituency_allocation_method}")

---

## Zusammenfassung

| Parameter                        | Typ                            | Standard              | Beschreibung                                                                           |
|----------------------------------|--------------------------------|-----------------------|----------------------------------------------------------------------------------------|
| `constituencies_config`          | `ConstituenciesConfig`         | —                     | Wahlkreise (Name, Größe)                                                               |
| `participating_parties`          | `list[str]`                    | —                     | Teilnehmende Parteien                                                                  |
| `parliament_majority_margin`     | `SuperMajorityMargin`          | 5 %                   | Regierungsabstand in % oder Sitzen                                                     |
| `ballot_majority_margin`         | `SuperMajorityMargin`          | 2 %                   | Wahlgangschwelle in % oder Sitzen                                                      |
| `constituency_representation`    | `ConstituencyRepresentation`   | `ENTIRE_PARLIAMENT`   | Wer bekommt Wahlkreise?                                                                |
| `draw_lots_strategy`             | `DrawLotsStrategy`             | `RANDOM`              | Gleichstandsauflösung                                                                  |
| `seed`                           | `int \| None`                  | `None`                | Startwert für Zufallsgenerator                                                         |
| `language`                       | `Language`                     | `Language.DE`         | Ausgabesprache für Tabellen und Diagramme                                              |
| `seat_distribution_method`       | `SeatDistributionMethod`       | `SAINTE_LAGUE`        | Standardmethode für Sitzverteilung; Evaluatorklassen können überschreiben              |
| `quota_correction_strategy`      | `QuotaCorrectionStrategy`      | `FAVOR_LARGE_PARTIES` | Standardstrategie für Quotenkorrektur; Evaluatorklassen können überschreiben           |
| `constituency_allocation_method` | `ConstituencyAllocationMethod` | `OPTIMAL`             | Standardalgorithmus für Wahlkreiszuweisung; Evaluatorklassen können überschreiben      |

### Abgeleitete Eigenschaften

| Eigenschaft                       | Beschreibung                                |
|-----------------------------------|---------------------------------------------|
| `parliamentarySeats`              | Gesamtzahl Parlamentssitze                  |
| `getParliamentMajoritySeats()`    | Sitze für Regierungsmehrheit                |
| `getParliamentMajorityPercent()`  | Mehrheitsschwelle in %                      |
| `governmentMajorityMarginSeats`   | Abstand in Sitzen (umgerechnet falls nötig) |
| `governmentMajorityMarginPercent` | Abstand in % (umgerechnet falls nötig)      |